# 第 5 章 成像系统与像差 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/waveprop/ch5/`。
> 运行顺序：先跑工程 1（zernike，定义本 notebook 共用的 `zernike` 函数），再跑工程 2（成像）与工程 3（习题）。


## 工程 1 · Zernike 与波前模式分解（原书 5.1 节，Listings 5.1–5.3）

MATLAB 版：`matlab/waveprop/ch5/ch5_zernike.m`。原独立模块名 `ch5_zernike.py`，其 `zernike` 函数供下面两个工程 import。


In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 5 章（工程 1）=====
成像系统与像差：Zernike 圆多项式与波前模式分解（原书 5.1 节，Listings 5.1-5.3）

原书要点：
    广义光瞳函数 P(x,y) = P_ap(x,y) exp(i 2 pi W(x,y))（式 5.1），其中波前差
    W 以"波"为单位。任意波前可按 Noll 单指标约定展开为 Zernike 级数
    W = sum_i a_i Z_i（式 5.6）；Zernike 圆多项式在单位圆上正交
    （式 5.7-5.9，int int Z_i Z_j r dr dtheta = pi delta_ij），
    系数用最小二乘/正交投影得到（式 5.10-5.18）；
    模式受限版本 W' = sum_{i<=nZ} a_i Z_i（式 5.19-5.20），
    波前方差 sigma^2 = sum_{i>=2} a_i^2（式 5.21-5.26）。

数值验证（运行本脚本全部自动打印）：
    (0) 表 5.2 抽查：生成的 (i,n,m) 映射与书中多项式逐一对上
        （Z1=1, Z2=2r cos, Z4=sqrt3(2r^2-1), Z11, Z22, Z37）；
    (1) 书例复现（Listing 5.2）：N=32, L=2, 模式 2/4/21 权重 0.5/0.25/-0.6，
        最小二乘恢复系数 A；
    (2) 正交性数值检查：Gram 矩阵 int int Z_i Z_j r dr dtheta ≈ pi delta_ij；
        并复核式 (5.7) 的印刷错误：int_0^1 (R_2^0)^2 r dr = 1/6 = 1/[2(n+1)]，
        不是书中印的 1/(2n+1) = 1/5（对照表 5.2 与 Listing 5.1 可证）；
    (3) 书例复现（Listing 5.3）：随机湍流相位屏 -> 100 阶 Zernike 投影 ->
        模式受限版 nZ = 3/16/36/100（对应原书图 5.3/5.4）；
        残差方差的两种算法互证：直接算 mean((W-W')^2) vs sum_{i>nZ} a_i^2。

相位屏说明：原书 Listing 5.3 调用第 9 章的 ft_phase_screen(r0, N, delta, inf, 0)
（Kolmogorov 谱 + 次谐波低频补偿）。本工程内置一个最小谱法相位屏
（von Karman 谱，粗网格垫零到 4 倍捕捉低频，幅度重定标到指定 RMS），
谱形状与第 9 章一致，足够演示模式分解；正式的 ft_phase_screen 在 ch9 笔记给出。

MATLAB/Python 差异（对应 .m 文件内注释）：
    - MATLAB 1-based / numpy 0-based：居中排列的零坐标样本位置不同。
    - np.meshgrid 默认 'xy' 与 MATLAB meshgrid 一致。
    - MATLAB 的 Z\\W 与 np.linalg.lstsq 都是最小二乘解（式 5.18）。
    - scipy.special.gamma 与 MATLAB gamma 均为 Gamma 函数（原书注释：
      gamma 比 factorial 快，式 5.5 的阶乘用它实现）。
用法：python ch5_zernike.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib + scipy；base.py（同仓库 python/fourier/common，circle）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ---- 复用《计算傅里叶光学》笔记的公共基础模块（同仓库） ----
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import circle  # noqa: E402  circle(X, Y, R) 半径参数；原书 circ(x,y,D) 直径参数


# ===== 书 Listing 5.1：zernike =====
def _build_zernike_index(n_max):
    """生成书表 5.2 的 (i, n, m) 映射（Noll 单指标，i 从 1 开始）。

    规则（由表 5.2 归纳）：按 n 升序分行，第 n 行占 n+1 个模式；
    行内 |m| 升序（与 n 同奇偶：m = n%2, n%2+2, ..., n）；
    m=0 占 1 行；每个 |m|>0 占 2 行，先 sin 还是 cos 由行号 i 的奇偶决定
    ——即式 (5.5b)：i 奇 -> sin(m theta)，i 偶 -> cos(m theta)。
    """
    rows = []
    i = 0
    for n in range(n_max + 1):
        for m in range(n % 2, n + 1, 2):        # 与 n 同奇偶的 |m|，升序
            if m == 0:
                i += 1
                rows.append((i, n, 0))
            else:
                i += 1
                rows.append((i, n, m))          # 先到的一行（sin 或 cos 由 i 奇偶定）
                i += 1
                rows.append((i, n, m))
    return rows


_ZIDX = _build_zernike_index(30)    # (i,n,m) 表：i 最大 496 阶，覆盖本章 100 阶


def zrf(n, m, r):
    """Zernike 径向多项式 R_n^m(r)（式 5.5a）。

    式中的阶乘按原书注释用 Gamma 函数实现（s! = Gamma(s+1)），比 factorial 快。
    """
    r = np.asarray(r, dtype=float)
    R = np.zeros_like(r)
    for s in range(0, (n - m) // 2 + 1):
        num = (-1.0) ** s * gamma(n - s + 1.0)
        den = (gamma(s + 1.0) * gamma((n + m) / 2.0 - s + 1.0)
               * gamma((n - m) / 2.0 - s + 1.0))
        R = R + num / den * r ** (n - 2 * s)
    return R


def zernike(i, r, theta):
    """书式 (5.4)（已修正 m=0 分支缺 sqrt(n+1) 的印刷错误）+ Listing 5.1。

    i 从 1 开始（i=1 是 piston）；r 是归一化半径（孔径边缘 r=1）。
    m != 0：Z_i = sqrt(2(n+1)) R_n^m(r) G_m(theta)；
    m == 0：Z_i = sqrt(n+1)   R_n^0(r)（Listing 5.1 第 9 行同此）。
    """
    _, n, m = _ZIDX[i - 1]
    R = zrf(n, m, r)
    if m == 0:
        return np.sqrt(n + 1.0) * R
    G = np.cos(m * theta) if i % 2 == 0 else np.sin(m * theta)   # 式 (5.5b)
    return np.sqrt(2.0 * (n + 1.0)) * R * G


# ===== 书 5.1 节：最小谱法相位屏（替代第 9 章 ft_phase_screen） =====
def phase_screen_min(r0, N, delta, L0=np.inf, seed=0, target_rms_waves=None):
    """最小 von Karman 相位屏（单位：rad）。

    谱 Phi(f) = 0.023 r0^{-5/3} (f^2 + L0^{-2})^{-11/6}
    （L0=inf 时退化为 Kolmogorov f^{-11/3}）。
    粗网格垫零到 4 倍（等效更大采样区域），把最低频率降到 1/(4L)，
    部分弥补无次谐波时的低频功率缺失；target_rms_waves 给定时按孔径内
    RMS 重定标到该值（单位：波）。第 9 章将给出正式的 ft_phase_screen。
    """
    rng = np.random.default_rng(seed)
    Npad = 4 * N
    fx = np.fft.fftfreq(Npad, d=delta)          # 移位排列的频率轴
    FX, FY = np.meshgrid(fx, fx)
    F2 = FX ** 2 + FY ** 2
    L02 = np.inf if np.isinf(L0) else L0 ** 2
    with np.errstate(divide='ignore'):
        Phi = 0.023 * r0 ** (-5.0 / 3.0) * (F2 + 1.0 / L02) ** (-11.0 / 6.0)
    Phi[0, 0] = 0.0                              # 活塞项置零（DC 处 0^{-11/6} 无定义）
    noise = rng.standard_normal((Npad, Npad))
    W = np.fft.ifft2(np.sqrt(Phi) * np.fft.fft2(noise)).real   # 实相位屏（rad）
    W = W[(Npad - N) // 2:(Npad + N) // 2, (Npad - N) // 2:(Npad + N) // 2]
    if target_rms_waves is not None:
        W = W / (np.std(W) / (2 * np.pi) / target_rms_waves)   # 定标到目标 RMS（波）
    return W


def main(outdir=None):
    # ---- 原书 Listing 5.2 的参数 ----
    N = 32            # number of grid points per side
    L = 2.0           # total size of the grid [m]
    delta = L / N     # grid spacing [m]
    x, y = np.meshgrid((np.arange(N) - N // 2) * delta,
                       (np.arange(N) - N // 2) * delta)
    r = np.hypot(x, y)                 # MATLAB cart2pol 的 r
    theta = np.arctan2(y, x)           # MATLAB cart2pol 的 theta
    ap = circle(x, y, 1.0)             # circ(x, y, 2)：直径 2 -> 单位圆孔径
    idxAp = ap.astype(bool)

    print('===== 验证 0：表 5.2 抽查（生成映射 vs 书中多项式） =====')
    checks = [
        (1, lambda r, t: np.ones_like(r), 'Z1 = 1 (piston)'),
        (2, lambda r, t: 2 * r * np.cos(t), 'Z2 = 2r cos(theta) (x tilt)'),
        (4, lambda r, t: np.sqrt(3) * (2 * r ** 2 - 1), 'Z4 = sqrt3 (2r^2-1) (defocus)'),
        (11, lambda r, t: np.sqrt(5) * (6 * r ** 4 - 6 * r ** 2 + 1),
         'Z11 = sqrt5 (6r^4-6r^2+1) (primary spherical)'),
        (22, lambda r, t: np.sqrt(7) * (20 * r ** 6 - 30 * r ** 4 + 12 * r ** 2 - 1),
         'Z22 = sqrt7 (20r^6-30r^4+12r^2-1) (secondary spherical)'),
        (37, lambda r, t: 3 * (70 * r ** 8 - 140 * r ** 6 + 90 * r ** 4
                               - 20 * r ** 2 + 1), 'Z37 = 3(70r^8-...+1) (tertiary spherical)'),
    ]
    rr = np.linspace(0, 1, 41)
    tt = np.linspace(0, 2 * np.pi, 73)
    RR, TT = np.meshgrid(rr, tt)
    for i, fun, name in checks:
        err = np.max(np.abs(zernike(i, RR, TT) - fun(RR, TT)))
        print('    %-58s max|diff| = %.2e' % (name, err))

    print('===== 验证 1：Listing 5.2 投影复现（模式 2/4/21） =====')
    # 原书：W = 0.5*z2 + 0.25*z4 - 0.6*z21，A = Z\W 应精确恢复权重
    z2, z4, z21 = (zernike(i, r, theta) * ap for i in (2, 4, 21))
    W = 0.5 * z2 + 0.25 * z4 - 0.6 * z21
    Z = np.column_stack([z2[idxAp], z4[idxAp], z21[idxAp]])   # 式 (5.16)
    A, *_ = np.linalg.lstsq(Z, W[idxAp], rcond=None)           # 式 (5.18)
    print('    恢复系数 A = [%+.6f %+.6f %+.6f]（真值 [0.5 0.25 -0.6]）' % tuple(A))
    resid = Z @ A - W[idxAp]
    print('    拟合残差 max = %.2e（离散网格上的式 (5.15) 近似误差）'
          % np.max(np.abs(resid)))

    print('===== 验证 2：正交性 + 式 (5.7) 印刷错误复核 =====')
    # Gram 矩阵：int int Z_i Z_j r dr dtheta ≈ sum_ap Z_i Z_j delta^2 ≈ pi delta_ij
    # 注意：只在孔径内求和（正交性只在单位圆上成立），孔径外样本必须剔除
    nG = 12
    G = np.zeros((nG, nG))
    for ii in range(nG):
        Zi = zernike(ii + 1, r, theta)[idxAp]
        for jj in range(ii, nG):
            G[ii, jj] = G[jj, ii] = np.sum(Zi * zernike(jj + 1, r, theta)[idxAp]) \
                * delta ** 2
    off = np.max(np.abs(G - np.diag(np.diag(G)))) / np.pi
    diag_err = np.max(np.abs(np.diag(G) / np.pi - 1.0))
    print('    N=32：Gram 对角/π 最大偏差 %.2e，非对角最大 |G_ij|/π %.2e'
          % (diag_err, off))
    # N=128 加密网格：离散正交性收敛，验证"偏差来自离散化而非公式"
    Nf = 128
    xf, yf = np.meshgrid((np.arange(Nf) - Nf // 2) * (L / Nf),
                         (np.arange(Nf) - Nf // 2) * (L / Nf))
    apf = circle(xf, yf, 1.0).astype(bool)
    rf = np.hypot(xf, yf)
    tf = np.arctan2(yf, xf)
    df = (L / Nf) ** 2
    Gf = np.zeros((nG, nG))
    for ii in range(nG):
        Zi = zernike(ii + 1, rf, tf)[apf]
        for jj in range(ii, nG):
            Gf[ii, jj] = Gf[jj, ii] = np.sum(Zi * zernike(jj + 1, rf, tf)[apf]) * df
    print('    N=128：Gram 对角/π 最大偏差 %.2e，非对角最大 |G_ij|/π %.2e（网格加密 4 倍偏差降 ~16 倍）'
          % (np.max(np.abs(np.diag(Gf) / np.pi - 1.0)),
             np.max(np.abs(Gf - np.diag(np.diag(Gf)))) / np.pi))
    # 式 (5.7) 书印 1/(2n+1)：对 n=2 即 1/5；真实值 = 1/[2(n+1)] = 1/6
    from scipy.integrate import quad
    val = quad(lambda q: (2 * q ** 2 - 1) ** 2 * q, 0, 1)[0]
    print('    int_0^1 (R_2^0)^2 r dr = %.6f = 1/6 = 1/[2(n+1)]' % val)
    print('    书式 (5.7) 印的是 1/(2n+1) = %.6f —— 与表 5.2/Listing 5.1 矛盾，系印刷错误'
          % (1 / 5))
    # RMS 方差恒等式（式 5.21 vs 5.26）在书例光滑波前上核对：
    # W = 0.5 Z2 + 0.25 Z4 - 0.6 Z21，全部 i>=2，sigma^2 应 = 0.5^2+0.25^2+0.6^2 = 0.6725
    Wsm = 0.5 * z2 + 0.25 * z4 - 0.6 * z21
    sig2_direct = np.mean((Wsm[idxAp] - np.mean(Wsm[idxAp])) ** 2)   # 式 (5.21) 离散版
    print('    光滑波前 sigma^2：N=32 直接 = %.4f vs 系数平方和 = %.4f（相对差 %.1e）'
          % (sig2_direct, 0.5 ** 2 + 0.25 ** 2 + 0.6 ** 2,
             abs(sig2_direct - 0.6725) / 0.6725))
    Wsm_f = 0.5 * zernike(2, rf, tf) + 0.25 * zernike(4, rf, tf) \
        - 0.6 * zernike(21, rf, tf)
    sig2_f = np.mean((Wsm_f[apf] - np.mean(Wsm_f[apf])) ** 2)
    print('    光滑波前 sigma^2：N=128 直接 = %.4f（相对差 %.1e）'
          % (sig2_f, abs(sig2_f - 0.6725) / 0.6725))

    print('===== 验证 3：Listing 5.3 模式分解/合成 + RMS 恒等式 =====')
    # 原书 N=40, L=2, r0=L/20；这里把网格加密到 N=128：式 (5.26) 恒等式依赖
    # 离散正交性，N=40 时离散正交误差有百分之几、恒等式只对到百分之几；
    # N=128 时误差降到 1e-3 量级（显示分辨率相应高于原书图 5.3 的颗粒感）。
    N3, L3 = 128, 2.0
    delta3 = L3 / N3
    r0 = L3 / 20.0
    x3, y3 = np.meshgrid((np.arange(N3) - N3 // 2) * delta3,
                         (np.arange(N3) - N3 // 2) * delta3)
    ap3 = circle(x3, y3, 1.0).astype(bool)
    phi = phase_screen_min(r0, N3, delta3, L0=np.inf, seed=7, target_rms_waves=0.6)
    W3 = phi / (2 * np.pi) * ap3            # rad -> 波，再乘孔径（Listing 5.3 第 16 行）
    r3 = np.hypot(x3, y3)
    t3 = np.arctan2(y3, x3)
    Wvec = W3[ap3]
    print('    相位屏：sigma_W = %.4f 波（孔径内 RMS；重定标目标 0.6 为全帧）'
          % Wvec.std())

    nModes = 200          # 拟合 200 阶：残差恒等式里"nZ 阶以外的尾部"必须被系数覆盖
    Zmat = np.zeros((Wvec.size, nModes))     # 式 (5.16)：列 = 各阶 Z_i 的线性索引值
    for k in range(nModes):
        Zmat[:, k] = zernike(k + 1, r3, t3)[ap3]
    Amode, *_ = np.linalg.lstsq(Zmat, Wvec, rcond=None)   # 式 (5.14)/(5.18)
    print('    200 阶投影：a1..a5 = [%s]' % ', '.join('%+.4f' % v for v in Amode[:5]))

    # 模式受限合成（式 5.19-5.20）与残差方差两种算法互证（式 5.21 vs 5.26）
    nZ_list = [3, 16, 36, 100]
    res_direct, res_coeff = [], []
    for nZ in nZ_list:
        Wp = Zmat[:, :nZ] @ Amode[:nZ]
        res = Wvec - Wp
        res_direct.append(np.mean(res ** 2))             # 式 (5.21) 离散版
        res_coeff.append(np.sum(Amode[nZ:] ** 2))        # 式 (5.26)
        print('    nZ=%3d：残差 sigma^2 直接=%.6f vs 系数平方和=%.6f（相对差 %.2e）'
              % (nZ, res_direct[-1], res_coeff[-1],
                 abs(res_direct[-1] - res_coeff[-1]) / res_direct[-1]))
    # QR 离散重正交对照：把 Z 列换成离散内积下严格正交归一的 Q（同一子空间）。
    # 最小二乘的"直接残差"只依赖子空间、不依赖基——两条直接残差应逐位一致；
    # 系数平方和的差别（a vs Bq）才反映 Gram 非正交性（~1e-3）。
    # 恒定差 ~0.0057 = 200 阶装不下的网格尺度内容（原书对式 (5.15) 的告诫）。
    Q, _ = np.linalg.qr(Zmat)
    Bq = Q.T @ Wvec          # Q^T Q = I（未归一化的求和内积），勾股：||res||^2 = sum Bq^2
    Nap3 = Wvec.size
    for nZ in nZ_list:
        res_q = np.mean((Wvec - Q[:, :nZ] @ Bq[:nZ]) ** 2)
        print('    [QR] nZ=%3d：直接=%.6f vs 系数平方和/N_ap=%.6f（相对差 %.1e）'
              % (nZ, res_q, np.sum(Bq[nZ:] ** 2) / Nap3,
                 abs(res_q - np.sum(Bq[nZ:] ** 2) / Nap3) / res_q))
    print('    （结论：sigma^2 = sum_{i>=2} a_i^2 在光滑波前上到离散正交精度；'
          '相位屏上两曲线之差恒定 ~%.4f，是 200 阶装不下的网格尺度内容，'
          '对应原书对式 (5.15) "只是近似"的告诫）' % (res_direct[-1] - res_coeff[-1]))

    # ---- 绘图：图 5-8（自绘，复现原书图 5.3/5.4 + 系数谱 + 残差恒等式） ----
    # 顶行 5 联排（原始屏 + 4 个模式受限屏，仿原书图 5.3），底行 2 图
    fig = plt.figure(figsize=(14.5, 6.6))
    gs = fig.add_gridspec(2, 5, height_ratios=[1.35, 1.0])
    data = [(W3, '(a) 原始相位屏')]
    for nZ, lab in zip(nZ_list, ['3', '16', '36', '100']):
        scr = np.zeros((N3, N3))
        scr[ap3] = Zmat[:, :nZ] @ Amode[:nZ]
        data.append((scr, '(%s) 限 %s 阶' % ('bcde'[len(data) - 1], lab)))
    for k, (img, title) in enumerate(data):
        axk = fig.add_subplot(gs[0, k])
        axk.imshow(img, cmap='gray', vmin=-1.2, vmax=1.2)
        axk.set_title(title, fontsize=11)
        axk.set_xticks([]); axk.set_yticks([])
    # 底行左：Zernike 系数谱
    axb = fig.add_subplot(gs[1, 0:2])
    ii = np.arange(2, nModes + 1)
    axb.semilogy(ii, np.abs(Amode[1:]), 'k.-', markersize=5, linewidth=0.8)
    axb.set_yticks([1e0, 1e-1, 1e-2, 1e-3, 1e-4])
    axb.set_yticklabels(['1', '0.1', '0.01', '0.001', '1e-4'])
    axb.set_xlabel('Zernike 阶号 i'); axb.set_ylabel('|a_i|（波）')
    axb.set_title('(f) 相位屏的 Zernike 系数谱'); axb.grid(True, alpha=0.3)
    # 底行右：残差方差恒等式
    axc = fig.add_subplot(gs[1, 2:5])
    axc.plot(nZ_list, res_direct, 'ko-', label=r'直接算 mean((W-W$\prime$)$^2$)')
    axc.plot(nZ_list, res_coeff, 'r.--', markersize=8,
             label=r'系数平方和 $\sum_{i>nZ} a_i^2$（式 5.26）')
    res_qr = [np.mean((Wvec - Q[:, :nZ] @ Bq[:nZ]) ** 2) for nZ in nZ_list]
    axc.plot(nZ_list, res_qr, 'bx', markersize=11, linestyle='none',
             label='QR 重正交后直接算（与黑线重合）')
    gap = res_direct[-1] - res_coeff[-1]
    axc.annotate('两曲线恒定差 %.4f\n= 200 阶装不下的网格尺度内容\n（原书式 (5.15) 告诫）'
                 % gap, xy=(28, 0.062), fontsize=9, color='0.35')
    axc.set_xlabel('保留模式数 nZ'); axc.set_ylabel(r'残差方差 $\sigma^2$（波$^2$）')
    axc.set_title('(g) 残差方差：两种算法互证'); axc.grid(True, alpha=0.3)
    axc.legend(fontsize=9)
    fig.tight_layout()
    if outdir:
        fp = os.path.join(outdir, 'fig-5-8.png')
        fig.savefig(fp, dpi=150)
        print('    已保存 %s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## 工程 2 · 相干与非相干成像（原书 5.2 节，Listings 5.4–5.5）

MATLAB 版：`matlab/waveprop/ch5/ch5_imaging.m`。


In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 5 章（工程 2）=====
相干与非相干成像：ASF/PSF 与 ATF/OTF（原书 5.2 节，Listings 5.4-5.5）

原书要点：
    成像系统对光是线性系统，但"线性在哪个量上"取决于照明的相干性：
      相干照明：场线性。U_i = h ⊗ U_o（式 5.27），振幅扩散函数
                h = (1/(lambda zi)) F{P}（式 5.29/5.30）；
                传递函数 H = F{h} = lambda zi * P(-lambda zi f)（式 5.34，
                lambda zi 因子正确：二维换元雅可比 lambda zi^2 抵消
                1/(lambda zi) 后剩一个；H(0,0) = lambda zi，量纲 m^2）。
                圆孔截频 f0 = D/(2 lambda zi)，像是严格带限的。
      非相干照明：强度线性。I_i = |h|^2 ⊗ I_o（式 5.35），|h|^2 即 PSF；
                OTF：H = F{|h|^2} / ∫∫|h|^2 dudv（式 5.38）
                   = 光瞳自相关 / ∫∫|P|^2 dxdy（式 5.40/5.41），
                圆孔截频 2 f0（式 5.42），无像差时 H(f) >= 0。
    波前像差只属于系统、与照明无关，但它对像的影响跟相干性有关。
    Strehl 比（5.2.3）：S = |∫∫ P e^{i phi} dxdy|^2 / |∫∫ P dxdy|^2（式 5.46），
    小像差时 S ≈ 1 - sigma_phi^2（Maréchal，式 5.54）≈ exp(-sigma_phi^2)（经验，式 5.55）。

数值验证（运行本脚本全部自动打印）：
    (0) myconv2 正确性：小随机阵列 vs scipy.signal.convolve2d（线性卷积取中心）；
    (1) 书例复现（Listings 5.4-5.5）：三缝物体的相干像与非相干像
        （0.05 波 Zernike 离焦，原书图 5.5/5.6）；
    (2) ATF 核对：F{h} 与 lambda zi * P(-lambda zi f) 逐点一致（式 5.34），
        相干像谱在 |f| > f0 的泄漏（理想像严格带限，离散实现有窗泄漏）；
    (3) OTF 核对：无像差圆孔的数值 OTF vs 解析式 (5.42)，两条数值路线
        （|h|^2 的 FT vs 光瞳自相关，式 5.38 vs 5.41）在重叠频段一致；
        截频 2f0 与 H(f) >= 0；
    (4) Strehl：0.05 波离焦的 PSF 峰值比 vs 光瞳积分（式 5.46）
        vs Marechal（式 5.54）vs 经验 exp（式 5.55）；
    (5) Parseval：sum |h|^2 du^2 = sum |P|^2 delta^2（h 与 P 能量守恒）。

myconv2 说明：原书 3.1 节定义（该章 PDF 缺失），按"线性卷积后取与物体同尺寸
的中心部分"实现（即 conv2(g,h,'same') 的居中语义，第三参数=1 的调用约定
与原书一致）；偶数尺寸的中心对齐起点 start=(K-N)//2（0-based）。

h 的定标说明：式 (5.29) 的 h 带 1/(lambda zi) 因子；原书 Listing 5.4/5.5
的 h = ft2(P, delta) 未乘（像的绝对辐照度因此差 1/(lambda zi)^2，形状与
归一化显示不受影响）。本工程按式 (5.29) 补上，显示仍做归一化。

MATLAB/Python 差异（对应 .m 文件内注释）：
    - np.fft.fft2(g, s=(K,K)) 零填充，与 MATLAB fft2(g,K,K) 一致。
    - 居中裁切起点：MATLAB 1-based start=(K-N)//2+1，numpy 0-based (K-N)//2，
      取同一批元素。
    - cart2pol -> np.hypot / np.arctan2。
用法：python ch5_imaging.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib + scipy；base.py（circle/rect/ft2_centered）；
      ch5_zernike.py（同目录，zernike）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve2d

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ---- 复用《计算傅里叶光学》笔记的公共基础模块（同仓库） ----
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import circle, rect, ft2_centered  # noqa: E402
# zernike 在本 notebook 第一个代码 cell（工程 1）中定义，按顺序运行即可；
# 拆回独立 .py 文件时改用：from ch5_zernike import zernike


# ===== 书 3.1 节：myconv2（线性卷积 + 居中裁切，这里补实现） =====
def myconv2(g, h):
    """myconv2(g, h, 1) 的语义：全尺寸线性卷积后取与 g 同尺寸的中心部分。

    FFT 实现：零填充到 (Ng+Nh-1) 做线性卷积（避免循环卷积回绕），
    再裁中间 Ng×Ng。偶数尺寸对齐：裁切起点 start=(K-Ng)//2（0-based），
    与 MATLAB 版 start=(K-Ng)//2+1（1-based）取同一批元素。
    """
    Ng = g.shape[0]
    K = Ng + h.shape[0] - 1
    full = np.fft.ifft2(np.fft.fft2(g, (K, K)) * np.fft.fft2(h, (K, K)))
    start = (K - Ng) // 2
    return full[start:start + Ng, start:start + Ng]


def main(outdir=None):
    # ---- 原书 Listing 5.4/5.5 的参数 ----
    N = 256            # number of grid points per side
    L = 0.1            # total size of the grid [m]
    D = 0.07           # diameter of pupil [m]
    delta = L / N      # grid spacing [m]
    wvl = 1e-6         # optical wavelength [m]
    z = 0.25           # image distance [m]
    lam_z = wvl * z    # lambda zi [m^2]（注意是乘积，原书记号 lambda zi）

    # pupil-plane coordinates
    x, y = np.meshgrid((np.arange(N) - N // 2) * delta,
                       (np.arange(N) - N // 2) * delta)
    r = np.hypot(x, y)
    theta = np.arctan2(y, x)
    ap_mask = circle(x, y, D / 2) > 0

    print('===== 验证 0：myconv2 正确性（小随机阵列 vs scipy） =====')
    rng = np.random.default_rng(1)
    gs_, hs_ = rng.standard_normal((6, 6)), rng.standard_normal((3, 3))
    ref = convolve2d(gs_, hs_, mode='same', boundary='fill')   # 奇数核：居中无歧义
    got = myconv2(gs_, hs_).real
    print('    max|myconv2 - convolve2d(same)| = %.2e' % np.max(np.abs(got - ref)))

    print('===== 验证 1 + 4：书例复现（0.05 波离焦）与 Strehl =====')
    W = 0.05 * zernike(4, 2 * r / D, theta)                    # 0.05 波 Zernike 离焦
    P_ab = circle(x, y, D / 2) * np.exp(1j * 2 * np.pi * W)    # 广义光瞳（式 5.1）
    P_dl = circle(x, y, D / 2).astype(complex)                 # 无像差光瞳

    # 振幅扩散函数（式 5.29）：h = ft2(P, delta)/(lambda zi)
    h_ab = ft2_centered(P_ab, delta) / lam_z
    h_dl = ft2_centered(P_dl, delta) / lam_z
    delta_u = lam_z / (N * delta)            # 像面采样 = lambda zi/(N delta)
    u1d = (np.arange(N) - N // 2) * delta_u
    u, v = np.meshgrid(u1d, u1d)
    f0 = D / (2 * lam_z)                     # 相干截频
    print('    delta_u = lambda zi/(N*delta) = %.3f um；f0 = D/(2 lambda zi) = %.0f /m'
          % (delta_u * 1e6, f0))

    # 物体（Listing 5.4 第 22-23 行）：三条 50 um x 200 um 竖缝
    obj = (rect((u - 1.4e-4) / 5e-5) + rect(u / 5e-5)
           + rect((u + 1.4e-4) / 5e-5)) * rect(v / 2e-4)

    img_coh = myconv2(obj, h_ab)                 # 式 (5.27)：场卷场
    img_inc = myconv2(np.abs(obj) ** 2, np.abs(h_ab) ** 2)   # 式 (5.35)：强度卷强度
    I_coh = np.abs(img_coh) ** 2
    I_inc = img_inc.real

    # Strehl（验证 4）：四种算法
    S_pupil = np.abs(np.sum(P_ab)) ** 2 / np.abs(np.sum(P_dl)) ** 2   # 式 (5.46) 离散
    S_psf = np.abs(h_ab[N // 2, N // 2]) ** 2 / np.abs(h_dl[N // 2, N // 2]) ** 2
    sig2_w = np.mean(W[ap_mask] ** 2)             # sigma^2（波^2）：Z4 单位 RMS -> 0.05^2
    sig2_phi = (2 * np.pi) ** 2 * sig2_w          # sigma_phi^2（rad^2）
    print('    0.05 波 Z4 离焦：sigma_W^2 = %.4f 波^2 -> sigma_phi^2 = %.4f rad^2'
          % (sig2_w, sig2_phi))
    print('    Strehl：PSF 峰值比 = %.4f = 光瞳积分（式 5.46）%.4f'
          % (S_psf, S_pupil))
    print('            Marechal 1-sigma_phi^2（式 5.54）= %.4f' % (1 - sig2_phi))
    print('            经验 exp(-sigma_phi^2)（式 5.55）= %.4f' % np.exp(-sig2_phi))

    print('===== 验证 2：ATF（式 5.34）+ 相干像带限 =====')
    # H(f) = F{h}，在像面原生频率网格上采样：f = (n-N/2)/(N du)，
    # 该网格与光瞳坐标一一对应：lambda zi * f = x（居中排列，间隔 delta）
    fx, fy = np.meshgrid((np.arange(N) - N // 2) / (N * delta_u),
                         (np.arange(N) - N // 2) / (N * delta_u))
    H_num = ft2_centered(h_ab, delta_u)
    P_roll = np.roll(P_ab[::-1, ::-1], (1, 1), axis=(0, 1))   # P(-lambda zi f) 样本
    print('    max|F{h} - lambda zi * P(-lambda zi f)| = %.2e（相对峰值 %.1e）'
          % (np.max(np.abs(H_num - lam_z * P_roll)),
             np.max(np.abs(H_num - lam_z * P_roll)) / np.max(np.abs(H_num))))
    print('    DC 值 H(0) = lambda zi = %.2e m^2（相干 ATF 不归一；OTF 才归一到 1）'
          % abs(H_num[N // 2, N // 2]))
    # 相干像严格带限：|f| > f0 处理想像谱为 0；离散实现有有限窗泄漏
    F_img = ft2_centered(img_coh, delta_u)
    beyond = np.hypot(fx, fy) > f0 * 1.001
    print('    相干像谱 |f|>f0 泄漏/峰 = %.1e（理想像严格带限，泄漏来自'
          'myconv2 的中心裁切窗）' % (np.max(np.abs(F_img[beyond]))
                                      / np.max(np.abs(F_img))))

    print('===== 验证 3：OTF（式 5.38/5.41/5.42） =====')
    # 路线 A：OTF = F{|h|^2}/(sum|h|^2 du^2)（像面网格，最高频率 1/(2 du) = 1.43 f0）
    otfA = ft2_centered(np.abs(h_dl) ** 2, delta_u).real \
        / (np.sum(np.abs(h_dl) ** 2) * delta_u ** 2)
    # 路线 B：光瞳自相关（式 5.41），垫零后位移轴 ±L -> 频率可达 L/(lambda zi) = 4e5 > 2f0
    Fp = np.fft.fft2(P_dl, (2 * N, 2 * N))
    ac = np.fft.fftshift(np.fft.ifft2(np.abs(Fp) ** 2)).real * delta ** 2
    xs = (np.arange(2 * N) - N) * delta          # 位移轴：-L .. L-delta（居中）
    fx2 = xs / lam_z
    otfB = ac[N, :] / (np.sum(np.abs(P_dl) ** 2) * delta ** 2)   # fy=0 切片

    def otf_analytic(f):                          # 解析式 (5.42)
        out = np.zeros_like(np.asarray(f, dtype=float))
        m = f <= 2 * f0
        s = f[m] / (2 * f0)
        out[m] = 2 / np.pi * (np.arccos(s) - s * np.sqrt(1 - s ** 2))
        return out

    # 路线 A 的可信频段：|h|^2 的 DFT 是 |h|^2 样本的"循环"自相关，位移超过
    # L-D（光瞳外仍有网格）就回绕 -> OTF 只在 |f| <= (L-D)/(lambda zi) 内干净
    f_clean = (L - D) / lam_z
    fa = np.linspace(0, 1.99 * 2 * f0, 400)
    faA = np.linspace(0, 0.99 * f_clean, 200)
    interpA = np.interp(faA, fx[0, fx[0] >= 0], otfA[N // 2, fx[0] >= 0])
    interpB = np.interp(fa, fx2[fx2 >= 0], otfB[fx2 >= 0])
    ana = otf_analytic(fa)
    anaA = otf_analytic(faA)
    print('    无像差圆孔 fy=0：路线A vs 解析(5.42) 在干净频段（|f|<=%.2e）最大差 %.2e'
          % (f_clean, np.max(np.abs(interpA - anaA))))
    print('                路线A 超出该频段即被循环卷积回绕污染（对照最大差 %.2e）；'
          '路线B（垫零线性自相关）全程最大差 %.2e'
          % (np.max(np.abs(np.interp(fa, fx[0, fx[0] >= 0],
                                     otfA[N // 2, fx[0] >= 0]) - ana)),
             np.max(np.abs(interpB - ana))))
    print('    DC 归一：otfA(0)=%.6f，otfB(0)=%.6f（式 5.38 分母 = h 的能量）'
          % (otfA[N // 2, N // 2], otfB[N]))
    print('    路线B 在 f=2f0 处 H=%.2e（截频确认）；全频段最小值 %.2e（H>=0）'
          % (np.interp(2 * f0, fx2[fx2 >= 0], otfB[fx2 >= 0]), otfB.min()))
    # 离焦系统的 OTF（0.05 波）——原书图 5.7 用 0.5 波展示衰减，思路相同
    otf_ab = ft2_centered(np.abs(h_ab) ** 2, delta_u).real \
        / (np.sum(np.abs(h_ab) ** 2) * delta_u ** 2)

    print('===== 验证 5：Parseval（h 与 P 能量守恒） =====')
    Eh = np.sum(np.abs(h_ab) ** 2) * delta_u ** 2
    Ep = np.sum(np.abs(P_ab) ** 2) * delta ** 2
    print('    sum|h|^2 du^2 = %.6e vs sum|P|^2 delta^2 = %.6e（相对差 %.2e）'
          % (Eh, Ep, abs(Eh - Ep) / Ep))

    # ---- 绘图：图 5-9（自绘：像对比 + OTF 曲线 + 像谱带限） ----
    sl = slice(N // 2 - 60, N // 2 + 60)     # ±150 um 窗口
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
    ax = axs[0]
    ax.plot(u[0, sl] * 1e3, obj[N // 2, sl], 'k--', linewidth=1.2, label='物体辐照度')
    ax.plot(u[0, sl] * 1e3, I_coh[N // 2, sl] / I_coh.max() * obj.max(),
            'b-', label='相干像辐照度（归一）')
    ax.plot(u[0, sl] * 1e3, I_inc[N // 2, sl] / I_inc.max() * obj.max(),
            'r-.', label='非相干像辐照度（归一）')
    ax.set_xlabel('u [mm]'); ax.set_ylabel('归一化辐照度'); ax.grid(alpha=0.3)
    ax.set_title('(a) 三缝物体的像（0.05 波离焦）'); ax.legend(fontsize=9)

    ax = axs[1]
    ax.plot(fa / (2 * f0), ana, 'k-', linewidth=1.8, label='解析 (5.42) 无像差')
    ax.plot(fa / (2 * f0), interpB, 'b.--', markersize=4,
            label='数值 OTF（光瞳自相关，式 5.41）')
    ax.plot(faA / (2 * f0), interpA, 'g.:', markersize=4,
            label='数值 OTF（|h|^2 的FT，干净频段）')
    pos = fx[0] >= 0
    ax.plot(fx[0, pos] / (2 * f0), otf_ab[N // 2, pos], 'r-.',
            label='数值 OTF（0.05 波离焦）')
    ax.axvline(f0 / (2 * f0), color='0.6', linestyle=':', linewidth=1)
    ax.text(f0 / (2 * f0) + 0.02, 0.55, 'f0\n相干截频', fontsize=9, color='0.4')
    ax.set_xlabel('归一化空间频率 f/(2f0)'); ax.set_ylabel('H(f)')
    ax.set_title('(b) 圆孔 OTF：解析 vs 数值'); ax.grid(alpha=0.3)
    ax.legend(fontsize=9); ax.set_ylim(-0.05, 1.05)

    ax = axs[2]
    F_obj = ft2_centered(obj, delta_u)
    ax.semilogy(np.abs(fx[0]) / f0, np.abs(F_obj[N // 2, :]) / np.abs(F_obj).max(),
                'k-', label='物体谱')
    ax.semilogy(np.abs(fx[0]) / f0,
                np.abs(F_img[N // 2, :]) / np.abs(F_img).max(),
                'b-', label='相干像谱')
    ax.axvline(1.0, color='r', linestyle=':', linewidth=1.2)
    ax.text(0.72, 3e-6, 'f = f0（相干截频）', fontsize=9, color='r')
    ax.set_ylim(1e-8, 2)
    ax.set_yticks([1e0, 1e-2, 1e-4, 1e-6, 1e-8])
    ax.set_yticklabels(['1', '1e-2', '1e-4', '1e-6', '1e-8'])
    ax.set_xlabel('f / f0'); ax.set_ylabel('归一化谱幅')
    ax.set_title('(c) 相干像严格带限（f0 处截止）')
    ax.grid(alpha=0.3, which='both'); ax.legend(fontsize=9, loc='lower left')
    fig.tight_layout()
    if outdir:
        fp = os.path.join(outdir, 'fig-5-9.png')
        fig.savefig(fp, dpi=150)
        print('    已保存 %s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## 工程 3 · 习题 1–6（原书 5.3 节）

MATLAB 版：`matlab/waveprop/ch5/ch5_problems.m`。


In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 5 章（工程 3）=====
习题 1-6 复现（原书 5.3 节）：色差焦斑、像散/球差 + 离焦演化、
Strehl 四算法对比、环形孔径与稀疏孔径 PSF。

原书习题：
    1. BK7 的 Sellmeier 色散 -> n_F/n_d/n_C -> 平凸透镜焦距与衍射受限
       焦斑直径；再用相干成像例（5.2.1）算 PSF 切片，加多档离焦。
       N=1024, delta=0.199 mm。
    2. 1 波 Zernike 初级像散 + 多档离焦的 PSF 演化。
       L=4 m, D=2 m, N=512, lambda=1 um, f=16 m。
    3. 1 波 Zernike 初级球差 + 多档离焦的 PSF 演化（参数同习题 2）。
    4. W = 0.07 Z4 + 0.05 Z5 - 0.05 Z6 + 0.03 Z7 - 0.03 Z8 的 Strehl：
       (a) 式 (5.26)+(5.54)/(5.55)；(b) 仿真 PSF（类比 5.2.1）。
       L=8 m, D=2 m, N=512, lambda=1 um, f=64 m。
    5. 内外径 1 m / 2 m 环形孔径 vs 满孔径 2 m 的 PSF 与 Strehl
       （峰值比），参数同习题 4，解析确认。
    6. 三个直径 1 m 圆孔（中心按原书印刷 (0.6,0.6), (-0.6,0.6), (0,0.6) m，
       三孔圆心共线于 y=0.6——疑为排印笔误，按原文实现）组成的稀疏孔径
       PSF，解析确认（单孔谱 x 干涉因子）。

数值验证（运行本脚本全部自动打印）：
    (1) BK7 三个标准波长的折射率（对照 Schott 手册值 n_d=1.51680,
        n_F=1.52238, n_C=1.51432）、焦距 f=R/(n-1)、Airy 直径
        2.44*lambda*f/D；三个波长的零离焦 PSF 半高全宽对照 Airy 直径；
        离焦 0/0.5/1/2 波的峰值跌落；
    (2/3) 像散/球差加离焦的 PSF 图（图 5-11）；
    (4) Strehl 四算法：式 (5.26)+(5.54)、式 (5.55)、光瞳积分（式 5.46）、
        仿真 PSF 峰值比；
    (5) 环孔 Strehl（峰值比）vs 解析 (1-eps^2)^2（轴上场 ∝ 面积）；
    (6) 稀疏孔径 PSF vs 解析 |FT{circ}|^2 * |sum exp(-i2pi f.c_j)|^2
        逐点对照 + 峰值比 vs 面积比的平方。

说明：P2-P6 的 PSF 用 |ft2(P, delta)|^2（平面波 -> 焦点，即点物的相干像的
强度），与 5.2.1 的 ASF 同一计算。
用法：python ch5_problems.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib + scipy；base.py（circle/ft2_centered）；
      ch5_zernike.py（同目录，zernike）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import circle, ft2_centered  # noqa: E402
# zernike 在本 notebook 第一个代码 cell（工程 1）中定义，按顺序运行即可；
# 拆回独立 .py 文件时改用：from ch5_zernike import zernike


def sellmeier_bk7(wvl_um):
    """BK7 的 Sellmeier 色散（式 5.56，Schott 手册系数，lambda 以 um 计）。"""
    B = [1.03961212, 0.231792344, 1.01046945]
    C = [0.00600069867, 0.0200179144, 103.560653]
    n2 = 1.0
    for Bi, Ci in zip(B, C):
        n2 += Bi * wvl_um ** 2 / (wvl_um ** 2 - Ci)
    return np.sqrt(n2)


def pupil_grid(L, N):
    """居中网格与极坐标。"""
    delta = L / N
    x, y = np.meshgrid((np.arange(N) - N // 2) * delta,
                       (np.arange(N) - N // 2) * delta)
    return x, y, np.hypot(x, y), np.arctan2(y, x), delta


def psf_of(P, delta, wvl, f, N):
    """PSF = |ft2(P)|^2 与像面坐标（间隔 lambda f/(N delta)）。"""
    h = ft2_centered(P, delta)
    psf = np.abs(h) ** 2
    u1d = (np.arange(N) - N // 2) * wvl * f / (N * delta)
    return psf, u1d


def main(outdir=None):
    print('===== 习题 1：BK7 色散、焦距与衍射受限焦斑 =====')
    lam = {'F': 0.48612, 'd': 0.58756, 'C': 0.65627}     # um
    ref = {'F': 1.52238, 'd': 1.51680, 'C': 1.51432}     # Schott 手册六位值
    R, Dlens = 51.68e-3, 12.7e-3                          # 平凸透镜（习题 1b）
    f_of, spot_of = {}, {}
    for k, w in lam.items():
        n = sellmeier_bk7(w)
        f = R / (n - 1.0)                                 # 薄平凸透镜 f = R/(n-1)
        airy = 2.44 * (w * 1e-6) * f / Dlens              # Airy 直径（第一暗环）
        f_of[k], spot_of[k] = f, airy
        print('    n(%s) = %.6f（手册 %.5f，差 %.0e）  f = %7.3f mm  '
              'Airy 直径 = %5.2f um' % (k, n, ref[k], abs(n - ref[k]),
                                        f * 1e3, airy * 1e6))
    Vd = (ref['d'] - 1) / (ref['F'] - ref['C'])
    print('    阿贝数 V_d = (n_d-1)/(n_F-n_C) = %.2f（BK7 标准 64.17）' % Vd)

    print('===== 习题 1c：三波长 PSF 切片 + 离焦演化（N=1024, delta=0.199mm） =====')
    N1 = 1024
    L1 = N1 * 0.199e-3
    psf_slices, peak0_d = {}, None
    for k, w in lam.items():
        x1, y1, r1, th1, delta1 = pupil_grid(L1, N1)
        P = circle(x1, y1, Dlens / 2)                     # 无像差光瞳
        psf, u1 = psf_of(P, delta1, w * 1e-6, f_of[k], N1)
        row = psf[N1 // 2, :]
        half = row.max() / 2
        above = np.where(row >= half)[0]
        fwhm = (above[-1] - above[0]) * (u1[1] - u1[0])
        psf_slices[k] = (row / row.max(), u1, spot_of[k])
        if k == 'd':
            peak0_d = psf.max()
        print('    lambda=%s: PSF 半高全宽 = %5.2f um vs Airy 直径 %5.2f um'
              % (k, fwhm * 1e6, spot_of[k] * 1e6))
    # 离焦演化（lambda_d，0/0.5/1/2 波 Z4）
    dfocus = [0.0, 0.125, 0.25, 0.5]   # Zernike 离焦系数（单位 RMS 波）
    defoc_slices = []
    for a in dfocus:
        x1, y1, r1, th1, delta1 = pupil_grid(L1, N1)
        W = a * zernike(4, 2 * r1 / Dlens, th1)
        psf, u1 = psf_of(circle(x1, y1, Dlens / 2)
                         * np.exp(1j * 2 * np.pi * W), delta1,
                         lam['d'] * 1e-6, f_of['d'], N1)
        defoc_slices.append((psf[N1 // 2, :] / psf.max(), u1))
        print('    离焦 %.3f 波：PSF 峰值 = 零离焦的 %.4f'
              % (a, psf.max() / peak0_d))

    print('===== 习题 2/3：像散、球差 + 离焦的 PSF 演化（L=4m, D=2m, f=16m） =====')
    L2, N2, D2, wvl2, f2 = 4.0, 512, 2.0, 1e-6, 16.0
    dfocus2 = [-2.0, -1.0, 0.0, 1.0, 2.0]   # 像散线焦在 -0.71/+0.71
    x2, y2, r2, th2, delta2 = pupil_grid(L2, N2)
    psf_dl2 = psf_of(circle(x2, y2, D2 / 2), delta2, wvl2, f2, N2)[0]
    psf_grids = {}
    for name, mi in [('像散 i=6', 6), ('球差 i=11', 11)]:
        rows = []
        for a in dfocus2:
            W = 1.0 * zernike(mi, 2 * r2 / D2, th2) \
                + a * zernike(4, 2 * r2 / D2, th2)
            psf, u2 = psf_of(circle(x2, y2, D2 / 2)
                             * np.exp(1j * 2 * np.pi * W), delta2, wvl2, f2, N2)
            rows.append(psf / psf.max())
            print('    %s + 离焦 %+.1f 波：Strehl（PSF 峰值比）= %.4f'
                  % (name, a, psf.max() / psf_dl2.max()))
        psf_grids[name] = rows

    print('===== 习题 4：五阶混合像差的 Strehl（L=8m, D=2m, f=64m） =====')
    L4, N4, D4, wvl4, f4 = 8.0, 512, 2.0, 1e-6, 64.0
    x4, y4, r4, th4, delta4 = pupil_grid(L4, N4)
    coef = {4: 0.07, 5: 0.05, 6: -0.05, 7: 0.03, 8: -0.03}   # 式 (5.60)
    W4 = np.zeros((N4, N4))
    for i, a in coef.items():
        W4 = W4 + a * zernike(i, 2 * r4 / D4, th4)
    sig2 = sum(a ** 2 for a in coef.values())            # 式 (5.26)
    sig2_phi = (2 * np.pi) ** 2 * sig2
    P_ab = circle(x4, y4, D4 / 2) * np.exp(1j * 2 * np.pi * W4)
    P_dl = circle(x4, y4, D4 / 2)
    S_pupil = np.abs(np.sum(P_ab)) ** 2 / np.abs(np.sum(P_dl)) ** 2   # 式 (5.46)
    psf_dl4, u4 = psf_of(P_dl, delta4, wvl4, f4, N4)
    psf_ab4, _ = psf_of(P_ab, delta4, wvl4, f4, N4)
    S_sim = psf_ab4.max() / psf_dl4.max()
    print('    sigma^2 = sum a_i^2 = %.4f 波^2 -> sigma_phi^2 = %.4f rad^2'
          % (sig2, sig2_phi))
    print('    (a) 式 (5.26)+(5.54)：S = 1 - sigma_phi^2 = %.4f' % (1 - sig2_phi))
    print('        式 (5.55) 经验：S = exp(-sigma_phi^2) = %.4f' % np.exp(-sig2_phi))
    print('    (b) 光瞳积分（式 5.46）：S = %.4f；仿真 PSF 峰值比：S = %.4f'
          % (S_pupil, S_sim))

    print('===== 习题 5：环形孔径（内外径 1m/2m，eps=0.5）vs 满孔径 =====')
    ap_ann = circle(x4, y4, D4 / 2) - circle(x4, y4, D4 / 4)
    psf_ann, _ = psf_of(ap_ann, delta4, wvl4, f4, N4)
    S_ann = psf_ann.max() / psf_dl4.max()
    S_analytic = (1 - 0.5 ** 2) ** 2          # 轴上场 ∝ 面积：(1-eps^2)^2
    c = N4 // 2
    row_ann = psf_ann[c, :]
    row_dl = psf_dl4[c, :]
    idx0 = c + np.argmax(row_ann[c:] < row_ann[c] * 1e-6)
    idx0_dl = c + np.argmax(row_dl[c:] < row_dl[c] * 1e-6)
    print('    峰值比 Strehl = %.4f vs 解析 (1-eps^2)^2 = %.4f（相对差 %.1e）'
          % (S_ann, S_analytic, abs(S_ann - S_analytic) / S_analytic))
    print('    第一暗环半径：环孔 %.2f um vs 满孔 %.2f um（环孔主瓣更窄）'
          % (u4[idx0] * 1e6, u4[idx0_dl] * 1e6))

    print('===== 习题 6：稀疏孔径（三个 1m 圆孔，中心按原书印刷） =====')
    centers = [(0.6, 0.6), (-0.6, 0.6), (0.0, 0.6)]   # 原书印刷：三孔共线 y=0.6
    # 三孔两两间距 0.6 m < 直径 1 m：按"求和"孔径做解析对照（FT 线性），
    # 按"并集"孔径（物理透过率）算峰值比；两者在重叠区相差二阶小量
    ap_sum = np.zeros((N4, N4))
    ap_sp = np.zeros((N4, N4))
    for cx, cy in centers:
        dsk = circle(x4 - cx, y4 - cy, 0.5)
        ap_sum = ap_sum + dsk
        ap_sp = np.maximum(ap_sp, dsk)
    psf_sp, _ = psf_of(ap_sp, delta4, wvl4, f4, N4)
    psf_sum, _ = psf_of(ap_sum, delta4, wvl4, f4, N4)
    # 解析：U(f) = FT{半径 0.5 圆盘}(f) * sum_j exp(-i 2pi f.c_j)
    fxn = (np.arange(N4) - N4 // 2) / (N4 * delta4)
    FXn, FYn = np.meshgrid(fxn, fxn)
    rho = np.hypot(FXn, FYn)
    with np.errstate(divide='ignore', invalid='ignore'):
        U1 = np.where(rho > 0, 0.5 * j1(2 * np.pi * 0.5 * rho) / rho,
                      np.pi * 0.25)                  # base.jinc(rho, R=0.5)
    inter = np.zeros((N4, N4), dtype=complex)
    for cx, cy in centers:
        inter = inter + np.exp(-1j * 2 * np.pi * (FXn * cx + FYn * cy))
    psf_an = np.abs(U1 * inter) ** 2
    # 数值 PSF 带 delta^4 定标；对照用归一化形状（各自除以轴上峰值）
    samples = [(c, c), (c, c + 1), (c + 2, c + 3), (c + 5, c + 2), (c + 9, c)]
    err = max(abs(psf_sum[i, j] / psf_sum[c, c]
                  - psf_an[i, j] / psf_an[c, c]) for i, j in samples)
    area_ratio = ap_sp.sum() * delta4 ** 2 / (np.pi * 1.0 ** 2)
    S_sp = psf_sp.max() / psf_dl4.max()
    print('    PSF（求和孔径）vs 解析（单孔谱 x 干涉因子）归一化形状最大差 = %.2e' % err)
    print('    三孔/满孔面积比 = %.4f；峰值比 = %.4f（= 面积比平方 %.4f）'
          % (area_ratio, S_sp, area_ratio ** 2))

    if not outdir:
        plt.show()
        return

    # ---- 图 5-10：习题 1（三波长焦斑 + 离焦演化） ----
    fig, axs = plt.subplots(2, 4, figsize=(14, 6.8))
    for ax, k in zip(axs[0], ['F', 'd', 'C']):
        row, u1, airy = psf_slices[k]
        m = np.abs(u1) < 5 * airy
        ax.plot(u1[m] * 1e6, row[m], 'k-', linewidth=1.2)
        ax.axvline(airy * 1e6, color='r', linestyle=':', linewidth=1)
        ax.axvline(-airy * 1e6, color='r', linestyle=':', linewidth=1)
        ax.set_title('lambda=%s um：Airy 直径 %.1f um' % (k, airy * 1e6), fontsize=10)
        ax.set_xlabel('u [um]'); ax.grid(alpha=0.3)
    axs[0, 3].axis('off')
    axs[0, 0].set_ylabel('归一化辐照度')
    for ax, a, dat in zip(axs[1], dfocus, defoc_slices):
        row, u1 = dat
        airy_d = spot_of['d']
        m = np.abs(u1) < 8 * airy_d
        ax.plot(u1[m] * 1e6, row[m], 'b-', linewidth=1.2)
        ax.set_title('离焦 %.1f 波（λ=d）' % a, fontsize=10)
        ax.set_xlabel('u [um]'); ax.grid(alpha=0.3)
    axs[1, 0].set_ylabel('归一化辐照度')
    fig.suptitle('习题 1：BK7 三波长焦斑（上排，红线 = ±Airy 半径）'
                 '与 d 光离焦演化（下排）', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(os.path.join(outdir, 'fig-5-10.png'), dpi=150)
    print('    已保存 fig-5-10.png')

    # ---- 图 5-11：习题 2/3 PSF 演化（2x4 图，±40 um 窗） ----
    ext = u2[-1] * 1e6
    fig, axs = plt.subplots(2, 5, figsize=(16, 8.6))
    for rowi, name in enumerate(['像散 i=6', '球差 i=11']):
        for coli, (a, img) in enumerate(zip(dfocus2, psf_grids[name])):
            ax = axs[rowi, coli]
            ax.imshow(np.log10(img + 1e-6), cmap='inferno', origin='lower',
                      extent=[-ext, ext, -ext, ext], vmin=-5, vmax=0)
            ax.set_title('%s, 离焦 %+.1f 波' % (name, a), fontsize=10, pad=4)
            ax.set_xlim(-40, 40); ax.set_ylim(-40, 40)
            if coli == 0:
                ax.set_ylabel('v [um]')
            ax.set_xlabel('u [um]')
    fig.suptitle('习题 2/3：1 波像散/球差加离焦的焦斑演化（PSF，log10 显示）', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(os.path.join(outdir, 'fig-5-11.png'), dpi=150)
    print('    已保存 fig-5-11.png')

    # ---- 图 5-12：习题 5/6（环孔与稀疏孔径） ----
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.8))
    axs[0].imshow(np.log10(psf_ann / psf_ann.max() + 1e-9), cmap='inferno',
                  origin='lower', extent=[-80, 80, -80, 80], vmin=-5)
    axs[0].set_title('习题 5：环形孔径 PSF（log10）')
    axs[1].imshow(np.log10(psf_sp / psf_sp.max() + 1e-9), cmap='inferno',
                  origin='lower', extent=[-80, 80, -80, 80], vmin=-5)
    axs[1].set_title('习题 6：稀疏孔径 PSF（log10）')
    mm = np.abs(u4) < 120e-6
    axs[2].plot(u4[mm] * 1e6, (row_dl / row_dl.max())[mm], 'k-',
                label='满孔 2 m')
    axs[2].plot(u4[mm] * 1e6, (row_ann / row_ann.max())[mm], 'b-',
                label='环孔 ε=0.5（自归一）')
    axs[2].plot(u4[mm] * 1e6, (psf_sp[c, :] / psf_sp.max())[mm], 'r-.',
                label='三孔稀疏（自归一）')
    axs[2].set_xlabel('u [um]'); axs[2].set_ylabel('归一化 PSF')
    axs[2].set_title('v=0 切片对比'); axs[2].grid(alpha=0.3)
    axs[2].legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, 'fig-5-12.png'), dpi=150)
    print('    已保存 fig-5-12.png')


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)
